# ATRI V2 续训 · Colab 一键版

在 **Qwen3-4B + 已有 pilot adapter** 上继续 SFT（不从头训、不用 resume_from_checkpoint）。

## 使用前
1. 本页面右上角 **Runtime → Change runtime type → T4 GPU**（免费版即可）
2. 准备好本地生成的 `atri_handoff_colab.zip`（约 42 MB，就在工作区 `colab/` 目录）
3. 依次运行下面的格子，**不要跳**

## 会发生什么
- 自动装 LLaMA-Factory + bitsandbytes（复用 Colab 自带 torch，不重装）
- 自动把 T4 不支持的 `bf16` 改成 `fp16`
- 下载 Qwen3-4B（约 8 GB，首次几分钟）
- 在 `./adapter` 上继续训 2 epoch（数据很小，训练只要几分钟）
- 最后把新 adapter 打包成 zip 让你下载


## 1. 确认 GPU 和 bf16 支持

T4 是 Turing 架构，**不支持 bf16**，所以后面必须用 fp16。这一格就是验证这个事实。

In [ ]:
import shutil, subprocess, torch

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("\u26a0\ufe0f  找不到 nvidia-smi：当前是 CPU 运行时")
    print("   请点 Runtime \u2192 Change runtime type \u2192 选 T4 GPU \u2192 Save，然后重连会话")

print("torch     :", torch.__version__)
print("CUDA 可用 :", torch.cuda.is_available())

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU       :", p.name)
    print("显存      :", round(p.total_memory / 1024**3, 1), "GB")
    print("bf16 支持 :", torch.cuda.is_bf16_supported(), "  <- T4 上应为 False")
else:
    raise SystemExit("\u274c 没有 GPU。先切换运行时，别继续往下跑")


## 2. 安装依赖

复用 Colab 自带的 torch，只补 LLaMA-Factory 和 bitsandbytes，避免重装 torch 拖慢。

In [ ]:
!pip install -q --upgrade "transformers>=4.51" "peft>=0.14" "trl>=0.15" "accelerate>=1.0" "datasets>=3.0" bitsandbytes
!pip install -q "llamafactory[metrics]"
!llamafactory-cli version

## 3. 上传交接包

把本地 `colab/atri_handoff_colab.zip`（42 MB）传到 Colab。

**推荐方式：左侧文件面板拖拽**
1. 点心侧边栏的文件夹图标
2. 进入 `/content`
3. 拖文件进去，等进度走完
4. 回来运行本格

本格会先找文件、再校验大小（43,707,071 字节），没有就提示你传，不会白跑后面的步骤。


In [ ]:
import os, glob, shutil

ZIP      = "atri_handoff_colab.zip"
EXPECTED = 43707071          # 字节，用来校验上传是否完整

if os.path.exists(ZIP) and os.path.getsize(ZIP) == EXPECTED:
    print("✅ 交接包已就位:", round(EXPECTED / 1024 / 1024, 1), "MB")
else:
    found = [p for p in glob.glob("/content/**/atri_handoff_colab.zip", recursive=True) if os.path.isfile(p)]
    if found:
        shutil.copy(found[0], ZIP)
        print("已从", found[0], "复制到当前目录")
    else:
        print("""
❌ 当前目录还没有 atri_handoff_colab.zip

上传方法（推荐拖拽，比弹窗稳定）：
  1. 点 Colab 左侧边栏的文件夹图标 📁
  2. 进入 /content 目录
  3. 把本地 colab/atri_handoff_colab.zip 直接拖进这个面板
  4. 等进度条走完，重新运行本格

如果拖拽也不行，可以试弹窗（有时会失效）：
  from google.colab import files
  files.upload()
""")
        raise SystemExit("等待上传")

size = os.path.getsize(ZIP)
assert size == EXPECTED, f"文件大小 {size} != {EXPECTED}，上传不完整，请重传"
print("✅ 校验通过:", size, "字节")

## 4. 解压并校验

关键点：`adapter_model.safetensors` 必须是 47,205,296 字节，否则说明上传不完整。

In [ ]:
import os, zipfile, shutil

dest = "/content/atri-handoff"
shutil.rmtree(dest, ignore_errors=True)
os.makedirs(dest, exist_ok=True)
with zipfile.ZipFile("atri_handoff_colab.zip") as z:
    z.extractall(dest)

need = ["adapter/adapter_model.safetensors", "adapter/adapter_config.json",
        "data/dataset_info.json", "data/v2/pilot/atri_v2_pilot_clean.jsonl",
        "sft_continue_from_pilot.yaml"]
for f in need:
    p = os.path.join(dest, f)
    print(("OK  " if os.path.exists(p) else "缺失 ") + f, os.path.getsize(p) if os.path.exists(p) else "")

size = os.path.getsize(f"{dest}/adapter/adapter_model.safetensors")
assert size == 47205296, f"adapter 大小不对: {size}, 上传可能不完整"
print("\n校验通过，大小", size, "字节")

## 5. 按显卡自动选择精度

- **L4 / G4 / A100 / H100**：支持 bf16，自动用 `bf16: true`
- **T4**：Turing 架构不支持 bf16，自动改成 `fp16: true`

不用手改，本格读完 GPU 属性自己决定。LoRA 结构（r=32 / alpha=64 / q_proj,v_proj）、4bit 量化、qwen3_nothink 模板一律不动。


In [ ]:
import re, torch

p = "/content/atri-handoff/sft_continue_from_pilot.yaml"
s = open(p, encoding="utf-8").read()

# 自动选精度：L4 / A100 / H100 支持 bf16；T4 只支持 fp16
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if use_bf16:
    s = s.replace("fp16: true", "bf16: true")
    print("GPU 支持 bf16 → 使用 bf16")
else:
    s = s.replace("bf16: true", "fp16: true")
    print("GPU 不支持 bf16（T4 等）→ 改用 fp16")

assert ("bf16: true" in s) != ("fp16: true" in s), "精度配置状态异常，检查 yaml"
open(p, "w", encoding="utf-8").write(s)

print("---- 关键配置 ----")
for line in s.splitlines():
    if re.search(r"(model_name_or_path|adapter_name_or_path|quantization_bit|lora_|dataset|template|cutoff_len|bf16|fp16|batch|epoch|output_dir)", line):
        print(line)

## 6. 开始续训

自动下载基座 + 训练。**输出看最后几行的 `train_loss`**。

In [ ]:
%cd /content/atri-handoff
!rm -rf runs
!llamafactory-cli train sft_continue_from_pilot.yaml 2>&1 | tail -60

## 7. 打包产物并下载

产物在 `runs/atri-companion-qwen3-4b-sft-v2-continue/`，只打包 adapter 文件（checkpoint 体积大，不带）。

In [ ]:
import os, glob, shutil
from google.colab import files

run_dir = "/content/atri-handoff/runs/atri-companion-qwen3-4b-sft-v2-continue"
print("产物目录:", os.listdir(run_dir) if os.path.isdir(run_dir) else "不存在！")

out = "/content/atri-v2-continue-adapter.zip"
shutil.rmtree("/content/atri-v2-continue-adapter", ignore_errors=True)
os.makedirs("/content/atri-v2-continue-adapter", exist_ok=True)
for name in ["adapter_model.safetensors", "adapter_config.json"]:
    src = os.path.join(run_dir, name)
    if os.path.exists(src):
        shutil.copy(src, "/content/atri-v2-continue-adapter/")
        print("已收集:", name, os.path.getsize(src))
shutil.make_archive("/content/atri-v2-continue-adapter", "zip", "/content/atri-v2-continue-adapter")
print("\n总计:", os.path.getsize(out)/1024/1024, "MB")
files.download(out)

## 排错

| 报错 | 原因 | 处理 |
|---|---|---|
| `this device does not support pure_bf16` | 还在用 bf16 | 重跑第 5 格，确认输出里有 `fp16: true` |
| `torch.cuda.OutOfMemoryError` | 显存不够 | 把 yaml 里 `cutoff_len: 1024` 改 `512`，其余不动 |
| `libbitsandbytes_cuda*.so not found` | bnb 装错版本 | `!pip install -q --force-reinstall bitsandbytes` 后重跑第 6 格 |
| `dataset not found` / `file_name` 找不到 | 路径不对 | 确认 `%cd /content/atri-handoff` 且 `data/dataset_info.json` 在 |
| `adapter size mismatch` | zip 上传不完整 | 重跑第 3 格重新上传 |
| 训练跑完 `train_loss` 没打印 | 日志被截断 | 看 `runs/.../trainer_state.json` 里的 `log_history` |

**注意**：不要给这个 adapter 用 `resume_from_checkpoint`，那是同一趟中断续跑用的；继续训就是靠 `adapter_name_or_path`。
